# 02. Xây dựng mô hình baseline

## 1. Mục tiêu và phương pháp đánh giá

### 1.1. Xác định mục tiêu dự đoán

Mục tiêu của mô hình là dự đoán giá bán `SalePrice` cho các căn nhà trong tập `test.csv`.

- **Tập train:** chứa đặc trưng và giá bán thực tế, dùng để huấn luyện và đánh giá mô hình.
- **Tập test:** chứa đặc trưng nhưng không có giá bán, dùng để tạo dự đoán sau khi lựa chọn mô hình.
- **File kết quả:** gồm hai cột `Id` và `SalePrice`, theo định dạng `sample_submission.csv`.

Trong quá trình phát triển, mô hình được đánh giá trên dữ liệu train trước khi dùng để dự đoán tập test.

### 1.2. Chọn thước đo sai số

Đồ án sử dụng **RMSE trên logarit của giá bán** làm thước đo chính,
phù hợp với cách đánh giá của bài toán House Prices trên Kaggle.

Thước đo so sánh logarit của giá dự đoán với logarit của giá thực tế:

$$
\text{RMSE}_{\log} =
\sqrt{\frac{1}{n}\sum_{i=1}^{n}
\left(\log(\hat{y}_i)-\log(y_i)\right)^2}
$$

Trong đó:

- \(y_i\): giá bán thực tế của căn nhà thứ \(i\).
- \(\hat{y}_i\): giá bán được mô hình dự đoán.
- \(n\): số căn nhà được đánh giá.

Giá trị RMSE trên logarit **càng nhỏ càng tốt**. Thước đo này quan tâm
đến sai lệch theo tỷ lệ giữa giá dự đoán và giá thực tế.

Ở notebook khám phá dữ liệu, `log1p(SalePrice)` đã được khảo sát như
một phương án biến đổi mục tiêu. `log1p(y)` và `log(y)` là hai phép
biến đổi khác nhau; khi tính thước đo cần áp dụng nhất quán theo
công thức đã chọn. Giá dùng trong phép logarit phải lớn hơn 0.

### 1.3. Chọn cách chia dữ liệu đánh giá

Sử dụng **5-fold cross-validation** trên tập train để đánh giá mô hình:

1. Chia tập train thành 5 phần.
2. Mỗi lượt dùng 4 phần để huấn luyện và phần còn lại để đánh giá.
3. Lặp lại 5 lượt để mỗi phần được dùng đánh giá một lần.
4. Tính sai số trung bình và độ lệch chuẩn của 5 lượt.

Các mô hình được so sánh bằng cùng cách chia và cùng thước đo
RMSE trên logarit của giá bán. Cần xử lý dữ liệu thiếu và mã hóa
đặc trưng bên trong từng lượt huấn luyện để tránh rò rỉ dữ liệu.

Tập `test.csv` chỉ được dùng để tạo dự đoán cuối cùng vì không có
giá bán thực tế để tính sai số.

### 1.4. Cách đọc kết quả đánh giá

Sau 5 lượt cross-validation, kết quả được trình bày bằng:

- **Sai số từng lượt:** RMSE trên logarit của giá ở mỗi phần đánh giá.
- **Sai số trung bình:** mức sai số trung bình của 5 lượt.
- **Độ lệch chuẩn:** mức độ thay đổi của sai số giữa 5 lượt.

RMSE trên logarit **càng nhỏ càng tốt**. Khi so sánh các mô hình,
ưu tiên xem sai số trung bình trên cùng cách chia dữ liệu và cùng
thước đo. Độ lệch chuẩn giúp nhận biết kết quả có thay đổi nhiều
giữa các lượt hay không; nó không bảo đảm mô hình sẽ ổn định trên
dữ liệu mới.

Điểm này được tính trên thang logarit, không phải sai số trực tiếp
theo đơn vị USD. Kết quả cross-validation chỉ là ước lượng khả năng
dự đoán và chưa phải điểm đánh giá trên tập test.

## 2. Đọc dữ liệu

Notebook đọc lại dữ liệu gốc từ `data/raw` để có thể chạy độc lập
với notebook khám phá dữ liệu.

- `train.csv`: dữ liệu dùng để huấn luyện và đánh giá.
- `test.csv`: dữ liệu dùng để tạo dự đoán cuối cùng.

In [1]:
from pathlib import Path

import pandas as pd

# Hỗ trợ chạy từ thư mục notebooks hoặc thư mục gốc của đồ án.
PROJECT_DIR = Path.cwd()

if not (PROJECT_DIR / "data" / "raw").is_dir():
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data" / "raw"

if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        "Không tìm thấy data/raw. Hãy mở notebook từ thư mục "
        "đồ án hoặc thư mục notebooks."
    )

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

print("Kích thước train:", train.shape)
print("Kích thước test:", test.shape)
print("Cột chỉ có trong train:", set(train.columns) - set(test.columns))
print("Cột chỉ có trong test:", set(test.columns) - set(train.columns))

display(train.head())

Kích thước train: (1460, 81)
Kích thước test: (1459, 80)
Cột chỉ có trong train: {'SalePrice'}
Cột chỉ có trong test: set()


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## 3. Chuẩn bị dữ liệu

### 3.1. Tách đặc trưng và biến mục tiêu

Tách `SalePrice` thành biến mục tiêu `y`. Các cột còn lại được dùng
làm đặc trưng đầu vào, ngoại trừ `Id` là mã định danh của căn nhà.

Giữ lại `Id` của tập test để sử dụng khi tạo file dự đoán sau này.

In [2]:
# Tách đặc trưng đầu vào và giá bán thực tế
X = train.drop(columns=["Id", "SalePrice"])
y = train["SalePrice"].copy()

# Chuẩn bị đặc trưng test và giữ lại ID cho file kết quả
X_test = test.drop(columns=["Id"])
test_ids = test["Id"].copy()

print("Kích thước X:", X.shape)
print("Kích thước y:", y.shape)
print("Kích thước X_test:", X_test.shape)
print("Các cột đầu vào train và test giống nhau:", X.columns.equals(X_test.columns))

Kích thước X: (1460, 79)
Kích thước y: (1460,)
Kích thước X_test: (1459, 79)
Các cột đầu vào train và test giống nhau: True


### 3.2. Phân nhóm đặc trưng số và phân loại

Chia các đặc trưng đầu vào thành hai nhóm để chuẩn bị tiền xử lý:

- **Đặc trưng số:** các giá trị biểu thị số lượng, diện tích, năm hoặc
  mức độ.
- **Đặc trưng phân loại:** các giá trị biểu thị nhóm hoặc loại.

Riêng `MSSubClass` được lưu dưới dạng số nhưng là mã phân loại, nên
được đưa vào nhóm phân loại. Cách phân nhóm còn lại dựa trên kiểu
dữ liệu hiện tại và có thể được điều chỉnh sau khi đánh giá baseline.

In [3]:
# MSSubClass là mã phân loại, dù được lưu dưới dạng số.
numeric_category_columns = ["MSSubClass"]

# Tạo bản sao để không thay đổi dữ liệu gốc.
X = X.copy()
X_test = X_test.copy()

for column in numeric_category_columns:
    X[column] = X[column].astype("string")
    X_test[column] = X_test[column].astype("string")

categorical_columns = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

numeric_columns = X.select_dtypes(
    include="number"
).columns.tolist()

print("Số đặc trưng số:", len(numeric_columns))
print("Số đặc trưng phân loại:", len(categorical_columns))
print("Tổng số đặc trưng:", len(numeric_columns) + len(categorical_columns))
print("\nĐặc trưng số:", numeric_columns)
print("\nĐặc trưng phân loại:", categorical_columns)

Số đặc trưng số: 35
Số đặc trưng phân loại: 44
Tổng số đặc trưng: 79

Đặc trưng số: ['LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold']

Đặc trưng phân loại: ['MSSubClass', 'MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Elect

### 3.3. Thiết kế xử lý giá trị thiếu và mã hóa

Xây dựng quy trình tiền xử lý ban đầu:

- Với đặc trưng số: điền giá trị thiếu bằng trung vị và chuẩn hóa.
- Với đặc trưng phân loại: điền giá trị thiếu bằng nhãn
  `Missing`, sau đó mã hóa one-hot.
- Các nhóm phân loại chưa xuất hiện trong dữ liệu huấn luyện
  được bỏ qua khi biến đổi dữ liệu mới.

Đây là cách xử lý đơn giản để tạo baseline. Khi đọc dữ liệu,
pandas đã gộp `None` và `NA` của `MasVnrType`; baseline này
chưa phân biệt lại hai mã đó.

Quy trình sẽ được đưa vào pipeline của mô hình. Trong mỗi lượt
cross-validation, các giá trị điền thiếu, thông số chuẩn hóa và
nhóm mã hóa chỉ được học từ phần dữ liệu huấn luyện của lượt đó.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Quy trình xử lý đặc trưng số
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Quy trình xử lý đặc trưng phân loại
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Áp dụng quy trình tương ứng cho từng nhóm cột
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

print("Đã tạo quy trình tiền xử lý.")

Đã tạo quy trình tiền xử lý.
